# Defter 19 — Year-block validation of the theory-guided network (pilot, v2)
**Theory-guided AI reveals how climate extremes reshape global tree growth** · Prof. Dr. İlker Ercanlı

**Ne yapar (TR):** Teori gömülü ağı (T_D+S) 1902–2024 halkalarının tamamıyla eğitir; yılların %25'i
(5 yıllık bloklar, tüm sahalardan birlikte) doğrulama için dışarıda tutulur. Bu pilot yalnızca **1. katı**
ve **1 tohumu** çalıştırır; amaç kodun çalıştığını görmek, süreyi ölçmek ve dört modeli karşılaştırmak:

| İş | Model | Ne sınanıyor |
|---|---|---|
| 1 | `teori_vpd` | Mevcut model (Liebig: sıcaklık, toprak nemi, VPD), yeni bölmeyle |
| 2 | `teori_vpd_opt` | + sıcaklık optimumu (aşırı sıcakta büyüme düşebilir) |
| 3 | `teori_vpd_opt_re` | + ağaç düzeyi kalıcı fark (rastgele etki, iklimsiz) |
| 4 | `iklimsiz_re` | Aynı model, iklim tepkisi kapalı → iklimin katkısı = 3 − 4 |
| 5 | `teori_vpd_opt_cwd_re` | 3 + iklimsel su açığı (CWD) Liebig terimi |
| 6 | `teori_vpd_opt_spei_re` | 3 + 12 aylık su dengesi indeksi (SPEI benzeri) terimi |
| 7 | `teori_vpd_opt_cwd_spei_re` | 3 + ikisi birlikte |

v2: erken durdurma görülmemiş yıl bloklarında; iklim sinyali için blok içi (yüksek frekans) korelasyon.

**Önce:** Defter 1 çalışmış ve `ITRDB_AI/TENSOR/veri_v2.npz` mevcut olmalı (Defter 2 ile aynı veri).
**Çalışma zamanı:** Çalışma zamanı → türünü değiştir → **L4 veya T4 GPU**, yüksek RAM. Sonra "Tümünü çalıştır".
Kesilirse yeniden "Tümünü çalıştır" yeterli; biten işler atlanır, yarım kalan son epoch'tan sürer.
**Çıktılar:** `ITRDB_AI/SONUC_19/` (her iş bir klasör) ve özet `ITRDB_AI/SONUC_19/P1_pilot_ozet_v2.csv`.
**Bitiş satırı:** `==== DEFTER 19 TAMAM ====`

In [ ]:
# --- Setup ---
import os, json, time, math
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ANA = '/content/drive/MyDrive/ITRDB_AI'
except ImportError:                                   # local test outside Colab
    ANA = os.environ.get('ITRDB_AI', '.')
CIKTI = os.path.join(ANA, 'SONUC_19'); os.makedirs(CIKTI, exist_ok=True)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEV, torch.cuda.get_device_name(0) if DEV == 'cuda' else '(no GPU: Runtime -> change runtime type -> GPU)')

AYAR = dict(
    batch=32768, max_epoch=60, sabir=5, lr=2e-3, lr_agac=1e-2, wd=1e-4,
    J=8, gizli=128, katman=2, emb=8,                 # site-parameter network: hidden units and layers (searched in stage 2)
    takson_dusur=0.10, desc_gurultu=0.2, r0_bas_cm=2.0, sifir_mm=0.005,
    blok_yil=5, n_kat=4, bolme_tohum=2026,           # year-block split: 5-year blocks, 4 folds of ~25% of years
    ic_blok_oran=0.10,                               # share of training blocks held out for early stopping (unseen years)
    surum='v2',                                      # v2: early stopping on unseen years; within-block (high-frequency) skill
    agac_onsel=0.3,                                  # prior SD of the tree effect (log scale)
    hizli_deneme=False,                              # True: small subset and 2 epochs, to check the notebook end to end
)
AYAR.update(json.loads(os.environ.get('D19_AYAR', '{}')))   # override only for local testing
if AYAR['hizli_deneme']: AYAR['max_epoch'] = min(AYAR['max_epoch'], 2)
t = lambda x, dt=None: torch.as_tensor(np.asarray(x), dtype=dt).to(DEV)

In [ ]:
# --- Data (identical preparation to Defter 2) ---
d = np.load(os.path.join(ANA, 'TENSOR', 'veri_v2.npz'))
soz = json.load(open(os.path.join(ANA, 'TENSOR', 'veri_v2_sozluk.json')))
YIL0 = soz['yil0']; S = len(soz['site_code']); NG = len(soz['genus']); NS = len(soz['species'])
NSER = len(d['series_site']); NTREE = int(d['series_tree'].max()) + 1

clim = d['clim']; soil = d['soil']; gE = d['gE']; NY = clim.shape[1]
YIL1 = YIL0 + NY - 1
# rings need the previous year's climate, so 1901 rings are dropped (as in Defter 2)
k = (d['ring_year'] > YIL0) & (d['ring_year'] <= YIL1)
R_ser = t(d['ring_series'][k], torch.long); R_yil = t(d['ring_year'][k].astype(np.int64), torch.long)
R_w = t(d['ring_rw'][k], torch.float32); R_rp = t(d['ring_rprev'][k], torch.float32) / 10   # cm
ser_site = t(d['series_site'], torch.long); ser_tree = t(d['series_tree'], torch.long)
R_site = ser_site[R_ser]; R_tree = ser_tree[R_ser]
site_genus = t(d['site_genus'], torch.long); site_species = t(d['site_species'], torch.long)
NR = len(R_w)

# monthly climate per site: T, log(1+P), soil moisture M, VPD, growing-season weight gE (+ CWD, SPEI-like, below)
ozel = np.stack([clim[..., 0], np.log1p(np.maximum(clim[..., 1], 0)), soil, clim[..., 3],
                 np.broadcast_to(gE[:, None, :], soil.shape)], -1).reshape(S, NY * 12, 5).astype(np.float32)
# --- additional water-stress variables (computed from the same CRU TS and bucket-model data; no download) ---
# climatic water deficit: CWD = PET - AET, with AET = PET x M / Mmax (supply-limited evapotranspiration of the
# VS-Lite bucket, Mmax = 0.76); in units of 100 mm per month
MMAX = 0.76
pet = np.maximum(np.nan_to_num(clim[..., 2]), 0)
cwd = (pet * (1 - np.clip(soil, 0, MMAX) / MMAX) / 100).reshape(S, NY * 12)
# SPEI-like index: 12-month sum of P - PET, standardized per site and calendar month against 1901-1960
wb = (np.nan_to_num(clim[..., 1]) - pet).reshape(S, NY * 12)
cs = np.cumsum(wb, axis=1); wb12 = np.full_like(cs, np.nan); wb12[:, 11:] = cs[:, 11:] - np.concatenate([np.zeros((S, 1)), cs[:, :-12]], 1)
wb12 = wb12.reshape(S, NY, 12); ref = wb12[:, :60]
z12 = (wb12 - np.nanmean(ref, 1, keepdims=True)) / (np.nanstd(ref, 1, keepdims=True) + 1e-6)
z12 = np.nan_to_num(np.clip(z12, -4, 4)).reshape(S, NY * 12)
ozel = np.concatenate([ozel, cwd[..., None].astype(np.float32), z12[..., None].astype(np.float32)], -1)
print(f'Water deficit: median {np.median(cwd)*100:.0f} mm/month; SPEI-like index range {z12.min():.1f} to {z12.max():.1f}')
del pet, cwd, wb, cs, wb12, ref, z12
n_nan = int(np.isnan(ozel).sum())
if n_nan: print(f'WARNING: {n_nan} missing climate values set to the site-month mean')
if n_nan:
    nk = ozel.shape[-1]
    ort = np.nanmean(ozel.reshape(S, NY, 12, nk), axis=1, keepdims=True)
    o4 = ozel.reshape(S, NY, 12, nk); o4[:] = np.where(np.isnan(o4), ort, o4); ozel = o4.reshape(S, NY * 12, nk)
OZ = t(ozel); del ozel

desc = d['desc']; DESC = t((desc - desc.mean(0)) / desc.std(0), torch.float32)
env = d['env']; ENV = t((env - env.mean(0)) / (env.std(0) + 1e-6), torch.float32)
site_biyom = t(d['site_biyom'], torch.long); site_eko = t(d['site_eko'], torch.long); site_kg = t(d['site_kg'], torch.long)
NB, NE, NK = soz['n_biyom'], soz['n_eko'], soz['n_kg']
IC = t(d['site_ic_dogrulama'], torch.bool)            # 10% of sites reserved for early stopping (fixed in Defter 1)
print(f'{NR:,} rings, {NSER:,} series, {NTREE:,} trees, {S} sites, years {YIL0+1}-{YIL1}')

def pencere(idx):
    """24-month climate window (previous January - current December) for rings idx: [B, 24, 5]"""
    s = R_site[idx]; bas = (R_yil[idx] - 1 - YIL0) * 12
    ay = bas[:, None] + torch.arange(24, device=DEV)[None, :]
    return OZ[s[:, None], ay], s

def girdiler(idx):
    X, s = pencere(idx)
    return dict(X=X, rp=R_rp[idx], tree=R_tree[idx], g=site_genus[s], sp=site_species[s], desc=DESC[s],
                env=ENV[s], b=site_biyom[s], e=site_eko[s], k=site_kg[s])

In [ ]:
# --- Year-block split: 5-year blocks; within each run of 4 consecutive blocks the folds are permuted,
#     so every fold covers the whole period (early, mid-century and the recent warm decades) ---
yillar = np.arange(YIL0 + 1, YIL1 + 1)
blok = (yillar - yillar[0]) // AYAR['blok_yil']; nb = int(blok.max()) + 1
rng = np.random.default_rng(AYAR['bolme_tohum']); kat_blok = np.empty(nb, int)
for b0 in range(0, nb, AYAR['n_kat']):
    ix = np.arange(b0, min(b0 + AYAR['n_kat'], nb)); kat_blok[ix] = rng.permutation(AYAR['n_kat'])[:len(ix)]
yk = np.full(YIL1 + 2, -1, np.int64); yk[yillar] = kat_blok[blok]
YK = t(yk, torch.long)
tablo = pd.DataFrame(dict(year=yillar, block=blok, fold=kat_blok[blok]))
tablo.to_csv(os.path.join(CIKTI, 'P0_yil_katlari.csv'), index=False)
R_kat = YK[R_yil]
for kk in range(AYAR['n_kat']):
    yy = tablo[tablo.fold == kk].year.to_numpy()
    print(f'fold {kk}: {len(yy)} years ({len(yy)/len(yillar):.0%}), {(R_kat == kk).float().mean().item():.1%} of rings; '
          f'blocks starting ' + ', '.join(str(y) for y in yy[::AYAR["blok_yil"]]))

bk = np.full(YIL1 + 2, -1, np.int64); bk[yillar] = blok
R_blok = t(bk, torch.long)[R_yil]

def bolunme(kat):
    """Test: this fold's years (~25%) at all sites. Early stopping: ~10% of the remaining 5-year blocks, drawn at
    random, at all sites (unseen years, as in the test). Train: all other years at all sites.
    No test year is used for training or for early stopping."""
    egit_blok = np.unique(blok[kat_blok[blok] != kat])
    n_ic = max(2, int(round(AYAR['ic_blok_oran'] * len(egit_blok))))
    ic_blok = np.random.default_rng(AYAR['bolme_tohum'] + 100 + kat).choice(egit_blok, n_ic, replace=False)
    test = R_kat == kat; ic = torch.isin(R_blok, t(ic_blok, torch.long))
    egit = (~test & ~ic).nonzero().squeeze(1); dog = (~test & ic).nonzero().squeeze(1); tst = test.nonzero().squeeze(1)
    print(f'fold {kat}: early-stopping blocks start ' + ', '.join(str(int(yillar[0] + b * AYAR['blok_yil'])) for b in sorted(ic_blok)))
    if AYAR['hizli_deneme']:
        g = torch.Generator(device='cpu').manual_seed(0)
        sec = lambda x, n: x[torch.randperm(len(x), generator=g)[:n].to(DEV)]
        egit, dog, tst = sec(egit, 200_000), sec(dog, 20_000), sec(tst, 50_000)
    return egit, dog, tst

In [ ]:
# --- Model: theory-embedded network (equations 1-3 of the manuscript) with options ---
class MLP(nn.Module):
    """Site-parameter network with `katman` hidden layers of `gizli` units (residual blocks beyond the first)."""
    def __init__(s, n_in, gizli, katman, n_out):
        super().__init__()
        s.gir = nn.Linear(n_in, gizli)
        s.bloklar = nn.ModuleList([nn.Sequential(nn.LayerNorm(gizli), nn.Linear(gizli, gizli), nn.GELU()) for _ in range(katman - 1)])
        s.cik = nn.Linear(gizli, n_out)
        with torch.no_grad(): s.cik.weight.mul_(0.1); s.cik.bias.zero_()
    def forward(s, x):
        h = F.gelu(s.gir(x))
        for b in s.bloklar: h = h + b(h)
        return s.cik(h)

class SiteAgi(nn.Module):
    """Genus/species embeddings + site descriptors + site environment (soil, topography, biome, ecoregion, Koppen)."""
    def __init__(s, n_out):
        super().__init__()
        e = AYAR['emb']
        s.ge = nn.Embedding(NG + 1, e); s.se = nn.Embedding(NS + 1, e)              # last index = unknown taxon
        s.b = nn.Embedding(NB + 1, 4); s.e = nn.Embedding(NE + 1, 4); s.k = nn.Embedding(NK + 1, 4)
        s.mlp = MLP(2 * e + DESC.shape[1] + ENV.shape[1] + 12, AYAR['gizli'], AYAR['katman'], n_out)
    def forward(s, G, gizle=None):
        g, sp, b, e, k = G['g'], G['sp'], G['b'], G['e'], G['k']
        if gizle is not None:                                                   # taxon dropout: learn the 'unknown' embedding
            g = torch.where(gizle, torch.full_like(g, NG), g); sp = torch.where(gizle, torch.full_like(sp, NS), sp)
        if s.training:                                                          # 10% masking of the categorical environment
            m = torch.rand(len(b), device=DEV) < 0.10
            b = torch.where(m, torch.full_like(b, NB), b); e = torch.where(m, torch.full_like(e, NE), e); k = torch.where(m, torch.full_like(k, NK), k)
        dn = G['desc'] + (AYAR['desc_gurultu'] * torch.randn_like(G['desc']) if s.training else 0)
        en = G['env'] + (AYAR['desc_gurultu'] * torch.randn_like(G['env']) if s.training else 0)
        return s.mlp(torch.cat([s.ge(g), s.se(sp), dn, en, s.b(b), s.e(e), s.k(k)], 1))

def monoton(x, a, c, s):
    """Equation (3): non-negative mixture of logistic functions, rising from 0 to 1. x:[B,24], a,c,s:[B,J]"""
    w = F.softplus(a) + 1e-4
    z = torch.sigmoid((x[..., None] - c[:, None, :]) / s[:, None, :])
    return (z * w[:, None, :]).sum(-1) / w.sum(-1, keepdim=True)

class TeoriModeli19(nn.Module):
    def __init__(s, vpd=True, opt=False, iklim=True, agac=False, cwd=False, spei=False):
        super().__init__()
        J = AYAR['J']; s.J, s.vpd, s.opt, s.iklim, s.agac, s.cwd, s.spei = J, vpd, opt, iklim, agac, cwd, spei
        n = 3 + 6 * J + 24 + (3 * J if vpd else 0) + (2 if opt else 0) + (3 * J if cwd else 0) + (3 * J if spei else 0) + 3 + 1
        s.ag = SiteAgi(n)
        s.Tbaz = torch.linspace(-5, 25, J, device=DEV); s.Mbaz = torch.linspace(0.02, 0.6, J, device=DEV)
        s.Dbaz = torch.linspace(0.2, 3.0, J, device=DEV)
        s.Wbaz = torch.linspace(0.05, 1.5, J, device=DEV); s.Zbaz = torch.linspace(-2.5, 1.5, J, device=DEV)
        s.u = nn.Parameter(torch.zeros(NTREE, device=DEV))                     # tree effect (used only when agac=True)
        s.m_logit = math.log((2 / 3) / (1 - 2 / 3))

    def parametreler(s, G, gizle=None):
        o = s.ag(G, gizle); J = s.J; p = {}
        p['eta'] = torch.exp(o[:, 0] + math.log(0.7))
        p['Bmax'] = torch.exp(o[:, 1] + math.log(1e4))                         # cm^2
        p['m'] = torch.sigmoid(o[:, 2] + s.m_logit)                            # starts at 2/3 (metabolic scaling)
        i = 3
        p['aT'], p['cT'], p['sT'] = o[:, i:i+J], s.Tbaz + 5 * torch.tanh(o[:, i+J:i+2*J]), 0.5 + 2 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        p['aM'], p['cM'], p['sM'] = o[:, i:i+J], s.Mbaz + 0.1 * torch.tanh(o[:, i+J:i+2*J]), 0.005 + 0.03 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        p['omega'] = torch.softmax(o[:, i:i+24], 1); i += 24
        if s.vpd:
            p['aD'], p['cD'], p['sD'] = o[:, i:i+J], s.Dbaz + 0.3 * torch.tanh(o[:, i+J:i+2*J]), 0.05 + 0.3 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        if s.cwd:   # climatic water deficit (100 mm/month); response decreases with deficit
            p['aW'], p['cW'], p['sW'] = o[:, i:i+J], s.Wbaz + 0.2 * torch.tanh(o[:, i+J:i+2*J]), 0.05 + 0.3 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        if s.spei:  # 12-month standardized water balance; response increases with wetter conditions
            p['aZ'], p['cZ'], p['sZ'] = o[:, i:i+J], s.Zbaz + 0.5 * torch.tanh(o[:, i+J:i+2*J]), 0.2 + 0.5 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        if s.opt:   # temperature optimum: growth declines above a site-specific upper threshold (starts near 25 °C)
            p['cH'], p['sH'] = 25 + 7 * torch.tanh(o[:, i]), 0.5 + 2 * F.softplus(o[:, i+1]); i += 2
        p['sig0'], p['sigC'], p['sigR'] = o[:, i] - 0.3, 0.1 * o[:, i+1], 0.1 * o[:, i+2]; i += 3   # heteroscedastic SD
        p['r0'] = F.softplus(o[:, i] + math.log(math.expm1(AYAR['r0_bas_cm']))); i += 1
        return p

    def forward(s, idx, gizle=None, G=None):
        G = girdiler(idx) if G is None else G
        X = G['X']; p = s.parametreler(G, gizle)
        r = p['r0'] + G['rp']; B = math.pi * r ** 2
        x = (B / p['Bmax']).clamp_min(1e-8) ** (1 - p['m'])
        Gp = p['eta'] * B.clamp_min(1e-6) ** p['m'] * F.softplus(20 * (1 - x)) / 20       # equation (1), smoothed
        T, M, D, gEk = X[..., 0], X[..., 2], X[..., 3], X[..., 4]
        if s.iklim:
            gT = monoton(T, p['aT'], p['cT'], p['sT'])
            if s.opt: gT = gT * (1 - torch.sigmoid((T - p['cH'][:, None]) / p['sH'][:, None]))
            gM = monoton(M, p['aM'], p['cM'], p['sM'])
            sinir = torch.minimum(gT, gM)
            gD = 1 - monoton(D, p['aD'], p['cD'], p['sD']) if s.vpd else torch.ones_like(gT)
            sinir = torch.minimum(sinir, gD)                                   # Liebig: min(gT, gM, gD), equation (2)
            if s.cwd: sinir = torch.minimum(sinir, 1 - monoton(X[..., 5], p['aW'], p['cW'], p['sW']))
            if s.spei: sinir = torch.minimum(sinir, monoton(X[..., 6], p['aZ'], p['cZ'], p['sZ']))
        else:
            gT = gM = gD = sinir = torch.ones_like(T)                          # climate-free ablation
        C = (p['omega'] * gEk * sinir).sum(1).clamp_min(1e-6)
        u = s.u[G['tree']] if s.agac else torch.zeros_like(C)
        BAI = Gp * C * torch.exp(u)
        w_hat = (torch.sqrt((B + BAI) / math.pi) - r) * 10                     # mm
        mu = torch.log(w_hat.clamp_min(1e-4))
        ls = p['sig0'] + p['sigC'] * torch.log(C) + p['sigR'] * torch.log(r)  # aleatoric SD varies with climate and size
        return mu, ls, dict(gT=gT, gM=gM, gD=gD, C=C, G=Gp)

def yap(ad):
    return TeoriModeli19(vpd=True, opt='_opt' in ad, iklim=not ad.startswith('iklimsiz'), agac=ad.endswith('_re'),
                         cwd='_cwd' in ad, spei='_spei' in ad).to(DEV)

In [ ]:
# --- Training and evaluation ---
LOG_Y_SIFIR = math.log(AYAR['sifir_mm']); LOG_ESIK = math.log(0.01)
def hedef(idx):
    w = R_w[idx]
    return torch.where(w > 0, torch.log(w.clamp_min(1e-6)), torch.full_like(w, LOG_Y_SIFIR))

def nll(mu, ls, y):
    """Gaussian NLL of log ring width; zero rings are left-censored below the 0.01 mm resolution."""
    sifir = y <= LOG_Y_SIFIR + 1e-6
    tam = 0.5 * ((y - mu) / torch.exp(ls)) ** 2 + ls + 0.9189385
    sans = -torch.special.log_ndtr((LOG_ESIK - mu) / torch.exp(ls))
    return torch.where(sifir, sans, tam).mean()

@torch.no_grad()
def tahmin(model, idx, bs=131072):
    model.eval(); MU, LS = [], []
    for i in range(0, len(idx), bs):
        mu, ls, _ = model(idx[i:i+bs]); MU.append(mu); LS.append(ls)
    model.train(); return torch.cat(MU), torch.cat(LS)

def yillik_sinyal(idx, mu, y):
    """Median site correlation between observed and predicted growth anomalies, averaged per site and year
    (sites with >= 8 test years). 'tum': each series centred on its test-period mean, so slow size/age trends and
    differences between blocks count. 'blok_ici': each series centred within each 5-year test block, which removes
    trends and levels and keeps only year-to-year (high-frequency, mainly climatic) variation."""
    nz = (y > LOG_Y_SIFIR + 1e-6).cpu().numpy()
    df = pd.DataFrame(dict(ser=R_ser[idx].cpu().numpy(), s=R_site[idx].cpu().numpy(), yil=R_yil[idx].cpu().numpy(),
                           b=R_blok[idx].cpu().numpy(), o=y.cpu().numpy(), p=mu.float().cpu().numpy()))[nz]
    sonuc = {}
    for ad, grup in [('tum', ['ser']), ('blok_ici', ['ser', 'b'])]:
        a = df[['s', 'yil']].copy()
        a['o'] = df.o - df.groupby(grup).o.transform('mean'); a['p'] = df.p - df.groupby(grup).p.transform('mean')
        sy = a.groupby(['s', 'yil'])[['o', 'p']].mean().reset_index()
        r = sy.groupby('s').apply(lambda g: g.o.corr(g.p) if len(g) >= 8 else np.nan, include_groups=False).dropna()
        sonuc[ad] = (float(r.median()), int(len(r)))
    return sonuc

def egit(kat, ad, tohum):
    is_ad = f'yil4{AYAR["surum"]}-{kat}__{ad}__s{tohum}'
    kl = os.path.join(CIKTI, is_ad); os.makedirs(kl, exist_ok=True)
    if os.path.exists(os.path.join(kl, 'bitti.json')):
        print('skipped (done):', is_ad); return
    torch.manual_seed(tohum); np.random.seed(tohum)
    egt, dog, tst = bolunme(kat)
    model = yap(ad)
    ag = [p for n, p in model.named_parameters() if n != 'u']
    opt = torch.optim.AdamW([{'params': ag, 'lr': AYAR['lr'], 'weight_decay': AYAR['wd']},
                             {'params': [model.u], 'lr': AYAR['lr_agac'] if model.agac else 0.0, 'weight_decay': 0.0}])
    kayit = os.path.join(kl, 'son.pt'); bas_ep, en_iyi, sabir, gecmis = 0, float('inf'), 0, []
    if os.path.exists(kayit):
        ck = torch.load(kayit, map_location=DEV)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
        bas_ep, en_iyi, sabir, gecmis = ck['epoch'] + 1, ck['en_iyi'], ck['sabir'], ck['gecmis']
        print(f'{is_ad}: resuming from epoch {bas_ep}')
    n_egt = len(egt); n_agac = int(torch.unique(R_tree[egt]).numel())
    for ep in range(bas_ep, AYAR['max_epoch']):
        t0 = time.time(); kar = egt[torch.randperm(n_egt, device=DEV)]; top = 0.
        for i in range(0, n_egt, AYAR['batch']):
            b = kar[i:i+AYAR['batch']]
            gz = torch.rand(len(b), device=DEV) < AYAR['takson_dusur']
            mu, ls, _ = model(b, gz)
            kv = nll(mu, ls, hedef(b))
            kayip = kv + ((model.u[R_tree[b]] ** 2).mean() / (2 * AYAR['agac_onsel'] ** 2) * len(b) / n_egt if model.agac else 0.)
            opt.zero_grad(set_to_none=True); kayip.backward()
            torch.nn.utils.clip_grad_norm_(ag, 5.0); opt.step(); top += kv.item() * len(b)
        mu, ls = tahmin(model, dog); v = nll(mu, ls, hedef(dog)).item()
        gecmis.append(dict(epoch=ep, egitim=top / n_egt, dogrulama=v, sn=time.time() - t0))
        print(f'{is_ad} ep {ep:2d}  train {top/n_egt:.4f}  early-stop NLL {v:.4f}  ({time.time()-t0:.0f} s)')
        if v < en_iyi - 1e-4:
            en_iyi, sabir = v, 0; torch.save(model.state_dict(), os.path.join(kl, 'en_iyi.pt'))
        else:
            sabir += 1
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), epoch=ep, en_iyi=en_iyi, sabir=sabir, gecmis=gecmis), kayit)
        if sabir >= AYAR['sabir']: break
    # test on the withheld years (best model)
    model.load_state_dict(torch.load(os.path.join(kl, 'en_iyi.pt'), map_location=DEV))
    mu, ls = tahmin(model, tst); y = hedef(tst); sg = torch.exp(ls)
    nz = y > LOG_Y_SIFIR + 1e-6; e = (mu - y)[nz]
    ys = yillik_sinyal(tst, mu, y); r_med, n_site = ys['tum']; r_hf, n_hf = ys['blok_ici']
    np.savez_compressed(os.path.join(kl, 'test_tahmin.npz'), idx=tst.cpu().numpy().astype(np.int64),
                        mu=mu.cpu().numpy().astype(np.float16), sigma=sg.cpu().numpy().astype(np.float16))
    ozet = dict(is_ad=is_ad, kat=kat, model=ad, tohum=tohum, n_egitim=n_egt, n_dogrulama=len(dog), n_test=len(tst),
                n_agac_egitimde=n_agac, epoch=len(gecmis), sn_epoch=float(np.mean([g['sn'] for g in gecmis])),
                rmse_log=float(e.pow(2).mean().sqrt()), bias_log=float(e.mean()),
                r2_log=float(1 - e.pow(2).mean() / y[nz].var()), nll=float(nll(mu, ls, y)),
                kapsama90=float(((y - mu).abs() < 1.645 * sg)[nz].float().mean()),
                yillik_r_medyan=r_med, yillik_r_n_saha=n_site, blok_ici_r_medyan=r_hf, blok_ici_r_n_saha=n_hf,
                en_iyi_dogrulama=en_iyi, ayar=AYAR, gecmis=gecmis)
    json.dump(ozet, open(os.path.join(kl, 'bitti.json'), 'w'), indent=1)
    os.remove(kayit)
    print(f'DONE {is_ad}: test RMSE(log) {ozet["rmse_log"]:.3f}, R2 {ozet["r2_log"]:.3f}, '
          f'90% coverage {ozet["kapsama90"]:.3f}, median site r {r_med:.3f} ({n_site} sites), within-block r {r_hf:.3f} ({n_hf} sites)')

In [ ]:
# --- Pilot job queue: fold 0, seed 0 ---
ISLER = [(0, 'teori_vpd', 0), (0, 'teori_vpd_opt', 0), (0, 'teori_vpd_opt_re', 0), (0, 'iklimsiz_re', 0),
         (0, 'teori_vpd_opt_cwd_re', 0), (0, 'teori_vpd_opt_spei_re', 0), (0, 'teori_vpd_opt_cwd_spei_re', 0)]
for kat, ad, tohum in ISLER:
    egit(kat, ad, tohum)

In [ ]:
# --- Summary ---
satir = []
for kat, ad, tohum in ISLER:
    f = os.path.join(CIKTI, f'yil4{AYAR["surum"]}-{kat}__{ad}__s{tohum}', 'bitti.json')
    if os.path.exists(f):
        o = json.load(open(f)); satir.append({k: v for k, v in o.items() if k not in ('ayar', 'gecmis')})
oz = pd.DataFrame(satir)
oz.to_csv(os.path.join(CIKTI, f'P1_pilot_ozet_{AYAR["surum"]}.csv'), index=False)
pd.set_option('display.width', 220)
print(oz[['model', 'n_egitim', 'n_dogrulama', 'n_test', 'epoch', 'sn_epoch', 'rmse_log', 'r2_log', 'kapsama90',
          'yillik_r_medyan', 'blok_ici_r_medyan', 'blok_ici_r_n_saha']].round(3).to_string(index=False))
if {'teori_vpd_opt_re', 'iklimsiz_re'} <= set(oz.model):
    a = oz.set_index('model')
    print(f"\nClimate contribution (same model with vs without climate): "
          f"R2 {a.r2_log['teori_vpd_opt_re'] - a.r2_log['iklimsiz_re']:+.3f}, "
          f"median site r {a.yillik_r_medyan['teori_vpd_opt_re'] - a.yillik_r_medyan['iklimsiz_re']:+.3f}, "
          f"within-block r {a.blok_ici_r_medyan['teori_vpd_opt_re'] - a.blok_ici_r_medyan['iklimsiz_re']:+.3f}")
    for m in ['teori_vpd_opt_cwd_re', 'teori_vpd_opt_spei_re', 'teori_vpd_opt_cwd_spei_re']:
        if m in a.index:
            print(f"{m:28s} vs teori_vpd_opt_re: R2 {a.r2_log[m] - a.r2_log['teori_vpd_opt_re']:+.3f}, "
                  f"within-block r {a.blok_ici_r_medyan[m] - a.blok_ici_r_medyan['teori_vpd_opt_re']:+.3f}")
print('==== DEFTER 19 TAMAM ====')